In [1]:
import pandas as pd
import numpy as np

In [2]:
#read csv
df= pd.read_csv('train_cleaned.csv')

In [10]:
#calculate renewal rate for responded vs not responded, Group by both campaign_type and campaign_response.
renewal_rate_r_vs_nr=df[df['campaign_contacted']==1].groupby(['campaign_type','campaign_response'])['renewal_status'].apply(lambda x:x.mean()*100).reset_index()
renewal_rate_r_vs_nr.columns = ['campaign_type', 'campaign_response', 'renewal_rate']
print(renewal_rate_r_vs_nr)

      campaign_type  campaign_response  renewal_rate
0        agent_call                0.0     50.104471
1        agent_call                1.0     81.365659
2             email                0.0     50.252489
3             email                1.0     80.772704
4  renewal_discount                0.0     50.486936
5  renewal_discount                1.0     81.278880


## What this means?
The goal should be maximising the number of customers who respond. Since agent calls have the highest response rate, they get the most customers into the 81% renewal zone. Email leaves the majority of customers in the 50% zone because most people ignore it.

In [13]:
#Revenue Retained per campaign type.
Revenue_retained = df[(df['campaign_contacted']==1)&(df['campaign_response']==1)&(df['renewal_status']==1)].groupby('campaign_type')['Annual_Premium'].sum().round(2).reset_index()
Revenue_retained.columns=['Campaign_Type','Revenue_Retained']
print(Revenue_retained)

      Campaign_Type  Revenue_Retained
0        agent_call       740225687.0
1             email       185548151.5
2  renewal_discount       385522405.0


### Campaign Cost

In [15]:
# Number of customers contacted per campaign type
contacts_per_campaign = df[df['campaign_contacted']==1].groupby('campaign_type')['id'].count().reset_index()
contacts_per_campaign.columns = ['Campaign_Type', 'Customers_Contacted']

# Cost per contact
cost_per_contact = {
    'agent_call': 15,
    'email': 1,
    'renewal_discount': 50
}

# Calculate total campaign cost
contacts_per_campaign['Cost_Per_Contact'] = contacts_per_campaign['Campaign_Type'].map(cost_per_contact)
contacts_per_campaign['Total_Campaign_Cost'] = contacts_per_campaign['Customers_Contacted'] * contacts_per_campaign['Cost_Per_Contact']

print(contacts_per_campaign)

      Campaign_Type  Customers_Contacted  Cost_Per_Contact  \
0        agent_call                51937                15   
1             email                51947                 1   
2  renewal_discount                38025                50   

   Total_Campaign_Cost  
0               779055  
1                51947  
2              1901250  


In [24]:
#calculating ROI based on total campaign cost for a type, comparing with revenue generated
roi_df = Revenue_retained.merge(contacts_per_campaign, on='Campaign_Type')
roi_df['ROI'] = ((roi_df['Revenue_Retained'] - roi_df['Total_Campaign_Cost']) / roi_df['Total_Campaign_Cost'] * 100).round(2)
display(roi_df)

,Campaign_Type,Revenue_Retained,Customers_Contacted,Cost_Per_Contact,Total_Campaign_Cost,ROI
0,agent_call,740225687.0,51937,15,779055,94915.84
1,email,185548151.5,51947,1,51947,357087.42
2,renewal_discount,385522405.0,38025,50,1901250,20177.31


## Campaign ROI Analysis -- Important Caveat

The ROI figures calculated in this analysis represent **gross return**, 
assuming full revenue attribution to the campaign. 

### Why the numbers look high
The campaign costs are small relative to the premium revenue retained:
- Agent call: $779,055 spent to retain $740,225,687
- Email: $51,947 spent to retain $185,548,151
- Renewal discount: $1,901,250 spent to retain $385,522,405

### What this means
ROI figures of 94,915%, 357,087%, and 20,177% are technically correct 
but overstated because we are giving the campaign 100% credit for every 
renewal. In reality, some customers would have renewed anyway without 
any outreach.

### The correct interpretation
A proper incremental ROI calculation would only count renewals that 
happened **because** of the campaign -- using the 30 percentage point 
renewal lift we identified. This would produce lower but more accurate ROI figures.

For this analysis, the ROI figures are best used as a **relative comparison** 
between campaign types rather than absolute measures of return.